# Transfer learning on CUB-200-2011

Run this notebook in Google Colab with a GPU runtime. It downloads the dataset through the Hugging Face datasets API, prepares the classification folders, fine-tunes a pretrained ResNet, and writes one CSV log for that run.

Dataset API:

- Train: https://huggingface.co/datasets/Multimodal-Fatima/CUB_train
- Test: https://huggingface.co/datasets/Multimodal-Fatima/CUB_test

The code below is the project Python, with preprocessing pointed at those datasets instead of a local CUB folder.


In [ ]:
%pip install -q datasets


## Run settings


In [ ]:
# Change these before running the notebook again.
# Each execution writes a new CSV under outputs/classification/run_logs/.
TRAIN_DATASET = "Multimodal-Fatima/CUB_train"
TEST_DATASET = "Multimodal-Fatima/CUB_test"
DATA_DIR = "classification_data"
BACKBONE = "resnet18"  # resnet18, resnet34, or resnet50
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
OUTPUT_DIR = "outputs/classification"
VALIDATION_PER_SPECIES = 6
SEED = 42
IMAGE_SIZE = 224


## `src/data/dataset.py`

Reads the prepared folders. The species folder name is the label. Training images are flipped at random. ImageNet normalization is required by the pretrained ResNet.


In [ ]:
from pathlib import Path

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def make_transform(train):
    steps = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if train:
        steps.append(transforms.RandomHorizontalFlip())
    steps.append(transforms.ToTensor())
    steps.append(transforms.Normalize(MEAN, STD))
    return transforms.Compose(steps)


class BirdDataset(Dataset):
    def __init__(self, root, class_names, train):
        self.transform = make_transform(train)
        labels = {name: index for index, name in enumerate(class_names)}
        self.samples = []
        for species in class_names:
            folder = Path(root) / species
            if not folder.is_dir():
                continue
            for path in sorted(folder.iterdir()):
                if path.suffix.lower() == ".jpg":
                    self.samples.append((path, labels[species]))
        if not self.samples:
            raise ValueError("No JPEG images in " + str(root))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            return self.transform(image.convert("RGB")), label


def classification_loaders(data_dir, batch_size):
    data_dir = Path(data_dir)
    if not (data_dir / "complete.txt").is_file():
        raise ValueError("Prepare the dataset before training.")

    names = set()
    for split in ("train", "val", "test"):
        folder = data_dir / split
        if not folder.is_dir():
            raise ValueError("Missing split folder: " + str(folder))
        for species in folder.iterdir():
            if species.is_dir():
                names.add(species.name)
    class_names = sorted(names)
    if not class_names:
        raise ValueError("No species folders in " + str(data_dir))

    datasets = {
        "train": BirdDataset(data_dir / "train", class_names, train=True),
        "val": BirdDataset(data_dir / "val", class_names, train=False),
        "test": BirdDataset(data_dir / "test", class_names, train=False),
    }
    for split, dataset in datasets.items():
        print(split, "images:", len(dataset), flush=True)

    loaders = {}
    for split, dataset in datasets.items():
        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=split == "train",
        )
    return loaders, class_names


## `src/models/transfer.py`

ImageNet ResNet with a new species classification layer.


In [ ]:
from torch import nn
from torchvision import models

BACKBONES = {
    "resnet18": (models.resnet18, models.ResNet18_Weights),
    "resnet34": (models.resnet34, models.ResNet34_Weights),
    "resnet50": (models.resnet50, models.ResNet50_Weights),
}


def build_model(backbone, num_classes, pretrained):
    if backbone not in BACKBONES:
        choices = ", ".join(BACKBONES)
        raise ValueError("Unknown backbone '" + backbone + "'. Choose " + choices + ".")
    builder, weights = BACKBONES[backbone]
    model = builder(weights=weights.DEFAULT if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


## `src/evaluation/metrics.py`

Accuracy, and macro precision, recall, and F1. Every species counts equally.


In [ ]:
import torch


def classification_metrics(predictions, targets, num_classes):
    predictions = torch.as_tensor(predictions, dtype=torch.long).reshape(-1)
    targets = torch.as_tensor(targets, dtype=torch.long).reshape(-1)
    matrix = torch.bincount(
        targets * num_classes + predictions,
        minlength=num_classes * num_classes,
    ).reshape(num_classes, num_classes)

    true_positive = matrix.diag().float()
    predicted = matrix.sum(dim=0).float()
    support = matrix.sum(dim=1).float()
    precision = torch.zeros(num_classes)
    recall = torch.zeros(num_classes)
    precision[predicted > 0] = true_positive[predicted > 0] / predicted[predicted > 0]
    recall[support > 0] = true_positive[support > 0] / support[support > 0]
    f1 = torch.zeros(num_classes)
    denominator = precision + recall
    f1[denominator > 0] = (
        2 * precision[denominator > 0] * recall[denominator > 0] / denominator[denominator > 0]
    )
    total = matrix.sum().item()
    return {
        "accuracy": true_positive.sum().item() / total,
        "precision_macro": precision.mean().item(),
        "recall_macro": recall.mean().item(),
        "f1_macro": f1.mean().item(),
        "confusion_matrix": matrix.tolist(),
    }


## `src/visualization/curves.py`


In [ ]:
def save_training_curves(history, destination):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    epochs = [row["epoch"] for row in history]
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in history], label="train")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="validation")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[1].plot(epochs, [row["train_accuracy"] for row in history], label="train")
    axes[1].plot(epochs, [row["val_accuracy"] for row in history], label="validation")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    figure.tight_layout()
    figure.savefig(destination, dpi=150)
    plt.close(figure)


## `src/data/preprocess.py` and `src/data/clean_captions.py`

These load the two Hugging Face datasets, save 224×224 RGB images, hold out 6 training images per species for validation, and clean the caption text. The official test split stays the test split.


In [ ]:
import random
from collections import defaultdict

REPLACEMENT = "\ufffd"


def clean_caption_lines(text):
    cleaned = []
    removed = 0
    for caption in str(text).splitlines():
        caption = caption.strip()
        if caption == "":
            removed = removed + 1
            continue
        if REPLACEMENT in caption:
            if len(caption.split()) != 1:
                removed = removed + 1
                continue
            caption = caption.replace(REPLACEMENT + REPLACEMENT, " ")
            if REPLACEMENT in caption:
                removed = removed + 1
                continue
        caption = caption.lower()
        caption = " ".join(caption.split())
        if caption == "" or caption == "this":
            removed = removed + 1
            continue
        if caption not in cleaned:
            cleaned.append(caption)
        else:
            removed = removed + 1
    return cleaned, removed


def species_folder(label_index, class_names):
    name = class_names[label_index].strip().replace(" ", "_")
    return str(label_index + 1).zfill(3) + "." + name


def save_record(record, class_names, split, output):
    label_index = int(record["label"])
    folder_name = species_folder(label_index, class_names)
    file_name = Path(str(record["file_name"])).name
    image_path = output / split / folder_name / file_name
    image_path.parent.mkdir(parents=True, exist_ok=True)
    image = record["image"].convert("RGB")
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    image.save(image_path, quality=95)

    cleaned, _ = clean_caption_lines(record["description"])
    if cleaned:
        caption_path = output / "captions" / split / folder_name / Path(file_name).with_suffix(".txt")
        caption_path.parent.mkdir(parents=True, exist_ok=True)
        caption_path.write_text("\n".join(cleaned) + "\n", encoding="utf-8")
    return len(str(record["description"]).splitlines()), len(cleaned), 0 if cleaned else 1


def prepare_from_api(output_dir=DATA_DIR):
    from datasets import load_dataset

    output = Path(output_dir)
    if (output / "complete.txt").is_file():
        print("Dataset already prepared:", output.resolve(), flush=True)
        return output
    if output.exists():
        raise ValueError("Incomplete output. Delete " + str(output) + " and run this cell again.")

    print("Loading", TRAIN_DATASET, flush=True)
    train_data = load_dataset(TRAIN_DATASET, split="train")
    print("Loading", TEST_DATASET, flush=True)
    test_data = load_dataset(TEST_DATASET, split="test")
    class_names = list(train_data.features["label"].names)
    if len(class_names) < 2:
        raise ValueError("The dataset did not provide species names.")

    groups = defaultdict(list)
    for index, label in enumerate(train_data["label"]):
        groups[int(label)].append(index)

    random.seed(SEED)
    chosen = {"train": [], "val": []}
    for label in sorted(groups):
        indexes = groups[label][:]
        if len(indexes) <= VALIDATION_PER_SPECIES:
            raise ValueError("Each species must have training images left over.")
        random.shuffle(indexes)
        chosen["val"].extend(indexes[:VALIDATION_PER_SPECIES])
        chosen["train"].extend(indexes[VALIDATION_PER_SPECIES:])

    sources = {"train": train_data, "val": train_data, "test": test_data}
    selections = {
        "train": chosen["train"],
        "val": chosen["val"],
        "test": list(range(len(test_data))),
    }
    caption_before = 0
    caption_after = 0
    missing_captions = 0
    for split, indexes in selections.items():
        for count, index in enumerate(indexes, start=1):
            before, after, missing = save_record(
                sources[split][index],
                class_names,
                split,
                output,
            )
            caption_before = caption_before + before
            caption_after = caption_after + after
            missing_captions = missing_captions + missing
            if count % 500 == 0 or count == len(indexes):
                print(split, "saved", count, "/", len(indexes), flush=True)

    summary = (
        "Source train: " + TRAIN_DATASET + "\n"
        "Source test: " + TEST_DATASET + "\n"
        "Saved captions: " + str(caption_after) + "\n"
        "Removed captions: " + str(caption_before - caption_after) + "\n"
        "Images without a usable caption: " + str(missing_captions) + "\n"
    )
    (output / "caption_cleaning.txt").write_text(summary, encoding="utf-8")
    (output / "complete.txt").write_text("Preprocessing finished.\n", encoding="utf-8")
    print(summary, flush=True)
    print("Ready for classification:", output.resolve(), flush=True)
    return output


## `src/training/trainer.py`

Fine-tunes the ResNet. The saved weights are the epoch with the highest validation macro F1. The test split is scored once after that choice.

Each run writes its own CSV. The file name contains the task, backbone, learning rate, batch size, seed, and the UTC start time, for example `classification_resnet18_lr0p001_bs32_seed42_20261030T120000Z.csv`.


In [ ]:
import csv
import json
from datetime import datetime, timezone

LOG_COLUMNS = [
    "run_started_utc",
    "backbone",
    "learning_rate",
    "batch_size",
    "seed",
    "phase",
    "epoch",
    "train_loss",
    "train_accuracy",
    "train_precision_macro",
    "train_recall_macro",
    "train_f1_macro",
    "val_loss",
    "val_accuracy",
    "val_precision_macro",
    "val_recall_macro",
    "val_f1_macro",
    "test_loss",
    "test_accuracy",
    "test_precision_macro",
    "test_recall_macro",
    "test_f1_macro",
]


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def run_log_path(config, started):
    stamp = started.strftime("%Y%m%dT%H%M%SZ")
    learning_rate = format(float(config["learning_rate"]), "g").replace(".", "p")
    name = (
        "classification_"
        + config["backbone"]
        + "_lr"
        + learning_rate
        + "_bs"
        + str(int(config["batch_size"]))
        + "_seed"
        + str(int(config["seed"]))
        + "_"
        + stamp
        + ".csv"
    )
    folder = Path(config["output_dir"]) / "run_logs"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / name


def append_log(path, row):
    new_file = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=LOG_COLUMNS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)


def blank_log(started, config, phase, epoch):
    row = {column: "" for column in LOG_COLUMNS}
    row["run_started_utc"] = started.strftime("%Y-%m-%dT%H:%M:%SZ")
    row["backbone"] = config["backbone"]
    row["learning_rate"] = config["learning_rate"]
    row["batch_size"] = config["batch_size"]
    row["seed"] = config["seed"]
    row["phase"] = phase
    row["epoch"] = epoch
    return row


def fill_metrics(row, prefix, loss, metrics):
    row[prefix + "_loss"] = round(float(loss), 6)
    row[prefix + "_accuracy"] = round(float(metrics["accuracy"]), 6)
    row[prefix + "_precision_macro"] = round(float(metrics["precision_macro"]), 6)
    row[prefix + "_recall_macro"] = round(float(metrics["recall_macro"]), 6)
    row[prefix + "_f1_macro"] = round(float(metrics["f1_macro"]), 6)


def run_epoch(model, loader, criterion, optimizer, device, train, num_classes):
    if train:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    seen = 0
    predictions = []
    targets = []
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
        total_loss = total_loss + loss.item() * images.size(0)
        seen = seen + images.size(0)
        predictions.append(logits.argmax(1).detach().cpu())
        targets.append(labels.detach().cpu())
    metrics = classification_metrics(torch.cat(predictions), torch.cat(targets), num_classes)
    return total_loss / seen, metrics


def train_model(config):
    torch.manual_seed(int(config["seed"]))
    device = pick_device()
    loaders, class_names = classification_loaders(config["data_dir"], int(config["batch_size"]))
    model = build_model(config["backbone"], len(class_names), pretrained=True)
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=float(config["learning_rate"]),
        momentum=0.9,
    )
    output = Path(config["output_dir"]) / config["backbone"]
    output.mkdir(parents=True, exist_ok=True)
    started = datetime.now(timezone.utc)
    log_path = run_log_path(config, started)
    print("Device:", device, flush=True)
    print("Saving checkpoint to:", output.resolve(), flush=True)
    print("Run log:", log_path.resolve(), flush=True)

    best_f1 = -1.0
    best_epoch = 0
    history = []
    num_classes = len(class_names)
    for epoch in range(1, int(config["epochs"]) + 1):
        train_loss, train_metrics = run_epoch(
            model, loaders["train"], criterion, optimizer, device, True, num_classes
        )
        val_loss, val_metrics = run_epoch(
            model, loaders["val"], criterion, optimizer, device, False, num_classes
        )
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_accuracy": train_metrics["accuracy"],
                "val_loss": val_loss,
                "val_accuracy": val_metrics["accuracy"],
                "val_f1_macro": val_metrics["f1_macro"],
            }
        )
        row = blank_log(started, config, "epoch", epoch)
        fill_metrics(row, "train", train_loss, train_metrics)
        fill_metrics(row, "val", val_loss, val_metrics)
        append_log(log_path, row)
        print(
            "epoch",
            epoch,
            "train loss",
            round(train_loss, 4),
            "acc",
            round(train_metrics["accuracy"], 4),
            "val loss",
            round(val_loss, 4),
            "acc",
            round(val_metrics["accuracy"], 4),
            "f1",
            round(val_metrics["f1_macro"], 4),
            flush=True,
        )
        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_epoch = epoch
            torch.save(
                {
                    "model": model.state_dict(),
                    "backbone": config["backbone"],
                    "class_names": class_names,
                    "epoch": epoch,
                },
                output / "best.pt",
            )

    checkpoint = torch.load(output / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    test_loss, test_metrics = run_epoch(
        model, loaders["test"], criterion, None, device, False, num_classes
    )
    test_row = blank_log(started, config, "test", best_epoch)
    fill_metrics(test_row, "test", test_loss, test_metrics)
    append_log(log_path, test_row)
    (output / "history.json").write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")
    curves_path = output / "curves.png"
    save_training_curves(history, curves_path)
    print("best epoch", best_epoch, "val macro F1", round(best_f1, 4), flush=True)
    print(
        "test acc",
        round(test_metrics["accuracy"], 4),
        "precision",
        round(test_metrics["precision_macro"], 4),
        "recall",
        round(test_metrics["recall_macro"], 4),
        "f1",
        round(test_metrics["f1_macro"], 4),
        flush=True,
    )
    print("Run log:", log_path.resolve(), flush=True)
    return output, log_path, curves_path


def predict_image(image_path, checkpoint_path):
    device = pick_device()
    image_path = Path(image_path)
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    class_names = checkpoint["class_names"]
    model = build_model(checkpoint["backbone"], len(class_names), pretrained=False)
    model.load_state_dict(checkpoint["model"])
    model.to(device)
    model.eval()
    with Image.open(image_path) as image:
        tensor = make_transform(False)(image.convert("RGB"))
    tensor = tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        probabilities = torch.softmax(model(tensor), dim=1)[0]
    count = min(5, len(class_names))
    scores, indices = torch.topk(probabilities, count)
    for score, index in zip(scores.tolist(), indices.tolist()):
        print(class_names[index] + "\t" + format(score, ".4f"))


## Prepare the data and train

Run the cells above first. This cell downloads the datasets, writes the folders, trains, and saves the CSV log. A second run reuses `classification_data` when `complete.txt` is already there, and still writes a new CSV.


In [ ]:
prepare_from_api(DATA_DIR)
output_dir, log_path, curves_path = train_model(
    {
        "data_dir": DATA_DIR,
        "backbone": BACKBONE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "output_dir": OUTPUT_DIR,
        "seed": SEED,
    }
)
from IPython.display import Image as NotebookImage
from IPython.display import display

display(NotebookImage(filename=str(curves_path)))
print(log_path.resolve())


## Predict one image

Set `IMAGE_PATH` to a bird photo, then run the cell. This is the prediction half of `src/cli.py`.


In [ ]:
IMAGE_PATH = ""  # example: classification_data/test/001.Black_footed_Albatross/some_bird.jpg
CHECKPOINT_PATH = str(Path(OUTPUT_DIR) / BACKBONE / "best.pt")
if IMAGE_PATH:
    predict_image(IMAGE_PATH, CHECKPOINT_PATH)
else:
    print("Set IMAGE_PATH, then run this cell again.")
